# Visual Distractions — YOLOv11 Training
**Rutgers MBS Externship | Collaborative Solutions | Summer 2026**

Run cells top to bottom. Sections:
1. Environment Setup
2. First-Time Data Setup (run once)
3. Verify Dataset
4. Train YOLOv11
5. Evaluate & Test
6. Save Best Weights

## 1 — Environment Setup

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone the repo (skip if already cloned)
import os
if not os.path.exists('/content/visual-distractions-yolov11'):
    !git clone https://github.com/YOUR_USERNAME/visual-distractions-yolov11.git /content/visual-distractions-yolov11
%cd /content/visual-distractions-yolov11

In [ ]:
# Install Ultralytics (YOLOv11)
!pip install ultralytics -q

import ultralytics
ultralytics.checks()  # prints GPU info + ultralytics version

## 2 — First-Time Data Setup
**Run this section ONCE.** It copies the F25 dataset into your Sum26 folder and creates the train/val split.

Skip to Section 3 if `datasets/` already exists in your Sum26 folder.

In [ ]:
# Update these paths before running
import sys
sys.path.insert(0, '/content/visual-distractions-yolov11/scripts')

# Override paths inside split_dataset before importing
import split_dataset

split_dataset.F25_IMAGES_TRAIN = "/content/drive/MyDrive/F25_VisualDist_Extern/ALLCODE-DOWNLOADTHISDIRECTLY/Project Materials/datasets/images/train"
split_dataset.F25_LABELS_TRAIN = "/content/drive/MyDrive/F25_VisualDist_Extern/ALLCODE-DOWNLOADTHISDIRECTLY/Project Materials/datasets/labels/train"
split_dataset.SUM26_DATASETS   = "/content/drive/MyDrive/Sum26_VisualDist_Extern/Visual_Distractions_YOLOv11/datasets"

# Recalculate derived paths after override
import os
split_dataset.IMG_TRAIN = os.path.join(split_dataset.SUM26_DATASETS, "images", "train")
split_dataset.IMG_VAL   = os.path.join(split_dataset.SUM26_DATASETS, "images", "val")
split_dataset.LBL_TRAIN = os.path.join(split_dataset.SUM26_DATASETS, "labels", "train")
split_dataset.LBL_VAL   = os.path.join(split_dataset.SUM26_DATASETS, "labels", "val")

split_dataset.main()

## 3 — Verify Dataset

In [ ]:
import os

DATASETS = "/content/drive/MyDrive/Sum26_VisualDist_Extern/Visual_Distractions_YOLOv11/datasets"

for split in ["train", "val"]:
    imgs   = len(os.listdir(f"{DATASETS}/images/{split}"))
    labels = len(os.listdir(f"{DATASETS}/labels/{split}"))
    print(f"{split:5s} — images: {imgs}, labels: {labels}")
    if imgs != labels:
        print(f"  WARNING: image/label count mismatch in {split}!")

## 4 — Train YOLOv11

In [ ]:
from ultralytics import YOLO

DATA_YAML   = "/content/visual-distractions-yolov11/configs/data.yaml"
RUNS_DIR    = "/content/drive/MyDrive/Sum26_VisualDist_Extern/Visual_Distractions_YOLOv11/runs"

# Load pretrained YOLOv11s (downloads ~18MB on first run)
model = YOLO('yolo11s.pt')

results = model.train(
    data      = DATA_YAML,
    epochs    = 50,
    imgsz     = 640,
    batch     = 16,
    name      = 'vd_yolo11_run1',
    project   = RUNS_DIR,
    device    = 0,       # GPU
    patience  = 15,      # early stopping
    lr0       = 0.0005,  # matches F25 hyperparameters
    optimizer = 'SGD',
    mosaic    = 1.0,
    save      = True,
    save_period = 10,    # checkpoint every 10 epochs
)

print(f"Training complete. Best weights: {results.save_dir}/weights/best.pt")

## 5 — Evaluate & Test

In [ ]:
# Validate on val set
metrics = model.val()
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision: {metrics.box.mp:.3f}")
print(f"Recall:    {metrics.box.mr:.3f}")

In [ ]:
# Quick inference test on a sample image
import glob
sample_imgs = glob.glob("/content/drive/MyDrive/Sum26_VisualDist_Extern/Visual_Distractions_YOLOv11/datasets/images/val/*.png")[:3]

for img in sample_imgs:
    results = model.predict(source=img, conf=0.25, save=False)
    print(f"\n{img.split('/')[-1]}:")
    for r in results:
        for box in r.boxes:
            cls  = model.names[int(box.cls)]
            conf = float(box.conf)
            print(f"  {cls}: {conf:.2f}")

## 6 — Save Best Weights to Sum26

In [ ]:
import shutil

RUNS_DIR  = "/content/drive/MyDrive/Sum26_VisualDist_Extern/Visual_Distractions_YOLOv11/runs"
DEST      = "/content/drive/MyDrive/Sum26_VisualDist_Extern/Visual_Distractions_YOLOv11/best.pt"

# Find most recent run
import glob, os
weight_files = glob.glob(f"{RUNS_DIR}/**/weights/best.pt", recursive=True)
latest = max(weight_files, key=os.path.getmtime)

shutil.copy(latest, DEST)
print(f"Saved best.pt → {DEST}")